# 01c · Validación de `ioc_utils` v1.2 sobre TODO el corpus — Grupo 17
**Cierre del Paso 2.** Antes de ejecutar, en `MyDrive/PI_G17/src/`:
1. Renombrar el `ioc_utils.py` actual a `ioc_utils_v1_0.py` (se conserva para diagnosticar las 6 fallas).
2. Subir los nuevos `ioc_utils.py` (v1.2) y `data_prep.py`; en `tests/` subir el nuevo `test_ioc_utils.py`.

**Criterios de cierre:** (1) 329 pruebas en verde en este entorno, (2) 0 textos con IOC vivo tras `defang_texto`
en las 4 fuentes de correo, (3) causa de las 6 fallas de v1.0 identificada (salida enmascarada, nada clicable).
Todo lo que se imprime pasa por `enmascarar()`: ningún IOC sale en forma utilizable.

In [1]:
# Celda 1: Entorno
import os, sys, json, re, mailbox, zipfile, subprocess, importlib.util
from pathlib import Path
from collections import Counter
import pandas as pd
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
V1   = BASE / "data" / "raw" / "g17-phishing-raw-v1"
COMP = BASE / "data" / "raw" / "g17-phishing-complemento-v1"
print(ENV, BASE)

Mounted at /content/drive
colab /content/drive/MyDrive/PI_G17


In [2]:
# Celda 2: Pruebas (criterio 1)
%pip install -q pytest
carpeta = next(d for d in (BASE / "tests", BASE / "test") if d.exists())
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(carpeta)],
                   capture_output=True, text=True, cwd=BASE)
PRUEBAS = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
print(PRUEBAS)

329 passed in 2.85s


In [3]:
# Celda 3: Módulos: v1.2 (actual) y v1.0 (solo diagnóstico)
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); return mod
ioc  = cargar("ioc_utils", BASE / "src" / "ioc_utils.py")
dp   = cargar("data_prep", BASE / "src" / "data_prep.py")
ruta_v10 = BASE / "src" / "ioc_utils_v1_0.py"
ioc_v10 = cargar("ioc_utils_v1_0", ruta_v10) if ruta_v10.exists() else None
assert "v1.2" in (ioc.__doc__ or ""), "src/ioc_utils.py no es la v1.2"
enmascarar = lambda s: s.replace(".", "·").replace(":", "˸").replace("/", "∕")
print("v1.2 OK | v1.0 disponible para diagnóstico:", ioc_v10 is not None)

v1.2 OK | v1.0 disponible para diagnóstico: True


In [4]:
# Celda 4: Iteradores por fuente (solo lectura del crudo)
def leer_csv(p):
    for enc in ("utf-8", "latin-1"):
        try: return pd.read_csv(p, encoding=enc, low_memory=False)
        except UnicodeDecodeError: pass
def textos_v1():
    for s in ["CEAS_08", "Enron", "Ling", "SpamAssasin", "Nazario", "Nigerian_Fraud"]:
        df = leer_csv(next(V1.rglob(f"{s}.csv")))
        for t in (df["subject"].fillna("").astype(str) + "\n" + df["body"].fillna("").astype(str)):
            yield f"kaggle/{s}", t
errores = Counter()
def textos_complemento():
    for zp in sorted((COMP / "phishing_pot").glob("*.zip")):
        z = zipfile.ZipFile(zp)
        for n in z.namelist():
            if n.lower().endswith(".eml"):
                try: d = dp.texto_de_email(z.read(n)); yield "phishing_pot", d["subject"] + "\n" + d["body"]
                except Exception: errores["phishing_pot"] += 1
    for f in sorted((COMP / "nazario_anual").glob("phishing-*")):
        for msg in mailbox.mbox(str(f)):
            try: d = dp.texto_de_email(msg); yield f"nazario/{f.name}", d["subject"] + "\n" + d["body"]
            except Exception: errores[f"nazario/{f.name}"] += 1

In [5]:
# Celda 5: Validación (criterios 2 y 3)
stats, diag_v10 = {}, []
for fuente, t in list(textos_v1()) + list(textos_complemento()):
    s = stats.setdefault(fuente, {"textos": 0, "ioc_vivo_v1_2": 0, "con_url": 0})
    s["textos"] += 1
    s["ioc_vivo_v1_2"] += bool(ioc.iocs_vivos(ioc.defang_texto(t)))
    s["con_url"] += bool(ioc.urls_para_features(t))
    if ioc_v10 is not None and fuente.startswith("kaggle/"):
        vivos = ioc_v10.iocs_vivos(ioc_v10.defang_texto(t))
        if vivos:
            out = ioc_v10.defang_texto(t); i = out.find(vivos[0])
            diag_v10.append({"fuente": fuente, "vivo": enmascarar(vivos[0][:60]),
                             "contexto": enmascarar(out[max(0, i - 50): i + 70])})
for s in stats.values():
    s["pct_con_url"] = round(100 * s.pop("con_url") / max(s["textos"], 1), 1)
total_vivos = sum(s["ioc_vivo_v1_2"] for s in stats.values())
print("Textos totales:", sum(s["textos"] for s in stats.values()), "| con IOC vivo (v1.2):", total_vivos)
pd.DataFrame(stats).T

Textos totales: 94566 | con IOC vivo (v1.2): 0


,textos,ioc_vivo_v1_2,pct_con_url
kaggle/CEAS_08,39154.0,0.0,67.5
kaggle/Enron,29767.0,0.0,0.0
kaggle/Ling,2859.0,0.0,0.0
kaggle/SpamAssasin,5809.0,0.0,79.2
kaggle/Nazario,1565.0,0.0,16.2
kaggle/Nigerian_Fraud,3332.0,0.0,36.7
phishing_pot,8614.0,0.0,92.9
nazario/phishing-2015,307.0,0.0,92.2
nazario/phishing-2016,494.0,0.0,84.2
nazario/phishing-2017,325.0,0.0,88.6


In [6]:
# Celda 6: RESUMEN
print("===== RESUMEN (01c) =====")
print(json.dumps({"entorno": ENV, "pruebas": PRUEBAS, "total_ioc_vivo_v1_2": total_vivos,
                  "errores_parseo": dict(errores), "por_fuente": stats, "diagnostico_v1_0": diag_v10},
                 indent=1, ensure_ascii=False))

===== RESUMEN (01c) =====
{
 "entorno": "colab",
 "pruebas": "329 passed in 2.85s",
 "total_ioc_vivo_v1_2": 0,
 "errores_parseo": {},
 "por_fuente": {
  "kaggle/CEAS_08": {
   "textos": 39154,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 67.5
  },
  "kaggle/Enron": {
   "textos": 29767,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 0.0
  },
  "kaggle/Ling": {
   "textos": 2859,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 0.0
  },
  "kaggle/SpamAssasin": {
   "textos": 5809,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 79.2
  },
  "kaggle/Nazario": {
   "textos": 1565,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 16.2
  },
  "kaggle/Nigerian_Fraud": {
   "textos": 3332,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 36.7
  },
  "phishing_pot": {
   "textos": 8614,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 92.9
  },
  "nazario/phishing-2015": {
   "textos": 307,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 92.2
  },
  "nazario/phishing-2016": {
   "textos": 494,
   "ioc_vivo_v1_2": 0,
   "pct_con_url": 84.2
  },
  "na